# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, following FAIR data principles.

### Dataset Source
This dataset is defined using the Croissant schema, providing structured metadata and record sets. It is accessible via the following Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's review available record sets, their fields, and related entity `@id`s. For all subsequent analysis, **entities are referenced by their `@id` field** per FAIR best practices and Croissant conventions.

We will list all record sets, their `@id`, and the available fields (with their `@id`s) in each record set.

In [ ]:
# Helper function to inspect all record sets and their field @ids
def print_record_sets_info(ds):
    print('Available record sets and their fields:')
    for rs in ds.record_sets:
        print(f"- RecordSet: {rs.name} (@id: {rs.id})")
        print("  Fields:")
        for f in rs.fields:
            print(f"    • Field: {getattr(f, 'name', '<unnamed>')} (@id: {f.id})")

print_record_sets_info(dataset)

## 3. Data Extraction
Load data from specific record set(s) into pandas DataFrame(s) for analysis. All entities are referenced by their `@id`.

First, we gather the `@id`s of record sets and load data for each.

In [ ]:
# Gather all record set @ids for tabular extraction
record_sets = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set_id])} records from RecordSet @id: {record_set_id}")

# As an example, list columns for the first record set
if record_sets:
    print(f"Available columns in the first record set (@id: {record_sets[0]}):")
    print(dataframes[record_sets[0]].columns.tolist())
    display(dataframes[record_sets[0]].head())

## 4. Exploratory Data Analysis (EDA)
Now let's perform some typical data wrangling tasks. 

> **Note:** For this demo, we will work with the first available record set and choose a numeric and a grouping field by their `@id` as revealed above. You should substitute these with the actual `@id`s relevant for your analysis.

We'll filter, normalize, and group by key attributes. All column and field access uses the `@id` name.

In [ ]:
# Define which record set to focus on
record_set_id = record_sets[0]  # Use first found; replace for other sets
df = dataframes[record_set_id]

# Explore column names (field @ids) for this record set, help user pick a numeric field
print("Columns (field @ids):", df.columns.tolist())

# For this dataset, let's heuristically choose a numeric column (e.g., 'Age' with field @id)
# You may customize the field @id below to match the actual table

# Try to guess a numeric field by scanning for likely column names
potential_numeric_columns = [col for col in df.columns if col.lower().startswith('age') or col.lower().endswith('age') or 'age' in col.lower()]
if potential_numeric_columns:
    numeric_field_id = potential_numeric_columns[0]
else:
    numeric_field_id = df.select_dtypes(include='number').columns[0] if len(df.select_dtypes(include='number').columns) else df.columns[0]

print(f"Selected numeric field @id: {numeric_field_id}")

# Filter: Keep only rows with age > 50 (as sample cutoff; adjust as needed)
threshold = 50
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalize the selected numeric field (Z-score)
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Choose a grouping field (e.g., 'Sex' or 'AnatomicalSite') if available
grouping_candidates = [col for col in df.columns if any(x in col.lower() for x in ['sex', 'gender', 'anatomical', 'site', 'location'])]
group_field_id = grouping_candidates[0] if grouping_candidates else df.columns[1]
print(f"Grouping by field @id: {group_field_id}")

# Perform group-by aggregation if suitable
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
    display(grouped_df)

## 5. Visualization
Visualize the distribution of the selected numeric field and inspect groupwise summaries.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by grouping field (if groups exist)
if group_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we loaded a FAIR-compliant clinical oncology dataset defined by a Croissant schema and explored its key tabular record sets using the `mlcroissant` Python library. By referencing all data entities by their unique `@id`, we ensured best practice in reproducibility and provenance. We performed filtering, normalization, grouping, and basic visualizations, preparing the data for downstream statistical or machine learning analysis.

**Next steps:** You may continue with advanced analyses, apply more complex feature engineering, or integrate the data into custom workflows, always referencing fields and record sets by their `@id` using `mlcroissant`.

[mlcroissant documentation](https://mlcommons.github.io/croissant/python)